In [2]:
import os
import sys
import pandas as pd
from IPython.display import display, Markdown
pd.set_option('display.max_rows', 100)
pd.set_option('display.min_rows', 20)

import collections
import csv
from difflib import SequenceMatcher
from math import ceil

In [82]:
scsnl_db=pd.read_csv('/Users/daelsaid/scsnl_docs/redcap/registration_form_rework_2023/SCSNLParticipantDatabase_remap.csv',dtype='str',encoding='utf-8')

temp_db=scsnl_db

output_filename_prefix='scsnl_ptp_db_reg_form_remap'

In [83]:
temp_db.rename(columns={'participant_psycfam___1':'participant_psycfam___1_Autism','participant_psycfam___2':'participant_psycfam___2_ADHD','participant_psycfam___3':'participant_psycfam___3_Dyslexia/Language disorders','participant_psycfam___4':'participant_psycfam___4_Dyscalculia/math disability','participant_psycfam___5':'participant_psycfam___5_Schizophrenia','participant_psycfam___6':'participant_psycfam___6_Bipolar Disorder','participant_psycfam___7':'participant_psycfam___7_Fragile X syndrome','participant_psycfam___8':'participant_psycfam___8_Epilepsy','participant_psycfam___11':'participant_psycfam___11_Major Depression','participant_psycfam___9':'participant_psycfam___9_Other developmental delay or intellectual disability','participant_psycfam___10':'participant_psycfam___10_Other psychiatric and neurological disorders','psyc_famrel1':'psyc_famrel1_Autism','psyc_famrel2':'psyc_famrel2_ADHD','psyc_famrel3':'psyc_famrel3_Dyslexia/Language disorders','psyc_famrel4':'psyc_famrel4_Dyscalculia/math disability','psyc_famrel5':'psyc_famrel5_Schizophrenia','psyc_famrel6':'psyc_famrel6_Bipolar Disorder','psyc_famrel7':'psyc_famrel7_Fragile X syndrome','psyc_famrel8':'psyc_famrel8_Epilepsy','psyc_famrel9':'psyc_famrel9_Other developmental delay or intellectual disability','psyc_famrel10':'psyc_famrel10_Other psychiatric and neurological disorders','psyc_famrel11':'psyc_famrel11_Major Depression','psyc_extra':'psyc_extra_general'},inplace=True)

first_order_relatives=temp_db[['record_id','psyc_famrel1_Autism','participant_psycfam___1_Autism','psyc_famrel2_ADHD','participant_psycfam___2_ADHD','psyc_famrel3_Dyslexia/Language disorders','participant_psycfam___3_Dyslexia/Language disorders','psyc_famrel4_Dyscalculia/math disability','participant_psycfam___4_Dyscalculia/math disability','psyc_famrel5_Schizophrenia','participant_psycfam___5_Schizophrenia','psyc_famrel6_Bipolar Disorder','participant_psycfam___6_Bipolar Disorder','psyc_famrel7_Fragile X syndrome','participant_psycfam___7_Fragile X syndrome','psyc_famrel8_Epilepsy','participant_psycfam___8_Epilepsy','psyc_famrel9_Other developmental delay or intellectual disability','participant_psycfam___9_Other developmental delay or intellectual disability','psyc_famrel11_Major Depression','participant_psycfam___11_Major Depression','psyc_famrel10_Other psychiatric and neurological disorders','participant_psycfam___10_Other psychiatric and neurological disorders','psyc_extra_general']]


if unspecified sibling or parent, a 1 was assigned under both parents and siblings

In [84]:
firstorder_relatives_dict= {
    'Sibling: Sister':["parents and siblings", "Self, siblings, child/parent", "sibling and parent", "sibling, parent", "sibling, self", "siblings, parents", " Sibling and parent", "All immediate members have it Mom, Dad , older sister and Younger Brother", "Mother/Father-Child", "Sibling and father", "Sibling, father", "sibling; father", "Brother, sister and father", "Father and Siblings", "sibling, mother", "Siblings and mother", "Siblings,aunts uncles,mother,grandmother", "Mother grandmother siblings", "MOTHER- CHILD", "Mother, sisters and brother", "Mother,siblings,grandma,mother", "brother,sister,mom,grandma,aunt", "Twin brother/sister/mother", "Her parent and her sibling", "his birth mom and siblings", ",sister,aunt, mom,brothers,grandma", "Self and sibling", "Sibling", "sibling ", "Sibling - anxiety", "Sibling and child", "Sibling with Down Syndrome ", "Sibling with temper dysregulation disorder and anxuety", "Sibling, child", "siblings", "Siblings ", "brother and sister", "Brother, Sister", "Brother/sister", "Callie is the middle child so her older sister and younger brother have dyslexia. Father also believes he has it but was never diagnosed", "sister and brother", "Sister-Brother", "Sister, Brother", "son, sibling", "Twin", "Twin ", "Grandma,siblings", "half sibling", "Fatehr, grandfather and sibling", "his siblings also FAS", " Siblings ", "2 sibilings - 1 sister, 1 brother", "My child", "My Child has ADHD.", "Older siblings", "Oldest sibling ", "Self, sister and dad", "Mother, father, sister", "half-sister, self", "Child, Father and Siseter with Dyslexia", "Sister and Father", "Sister, father", "sister, likely father undiagnosed", "Father/daughter", "Father - Daughter", "Father and sisters anxiety and depression ", "biological father, sister", "My three daughthers and their dad has dyslexia.", "Mother, sister", "Mother, sister ", "Sister, likely mom", "Sister, mother", "Auntie mom sister ", "Mother/Daughter", "Siblings, oldest Sister Mild , Youngest sister Severe", "sister", "Sister ", "Child has it, sister has it.", "Sister - global developmental delays ", "sister and daughter", "Sister appears to have ADHD", "Sister daughter ", "Sister has Down syndrome/ASD/ADHD", "sister/uncle", "Sisters", "Twin sister", "Twin sister and first cousin", "Youngest sister", "Grandmother, Sister both have Cognitive Memory Disorder", "Half sister", "half-sister", "Half-Sister ", "Daughter", "daughter ", "Daughter of", "Daughter, sister", "Elder Sister", "Lexi's older sister recently got an informal diagnosis of ADHD by her GP.   She has not had any treatment as yet.", "Maternal cousin, sister", "1/2 sister", "Nephew and sister ", "older sister"],
    
    'Parent: mother':["Mom,uncle", "Parent w/ADHD", "Parent-child", "Parent, Uncle", "Parent/child", "Parents", "parents and siblings", "parents anxiety and depression", "Self, siblings, child/parent", "Self, sister and dad", "sibling and parent", "sibling, parent", "sibling, self", "siblings, parents", "mother - past mild depression & father - mild anxiety", "Mother & father", "mother & father major depression major anxitey", "mother and brother, anxiety.  father and mother, depression", "Mother and father", "Mother and father ", "mother and father Depression/ Anxiety", "Mother, father, sister", "Brother, Father, Mother", "Child both parents have adhd and older brother", "Child both parents have it", "Child/parent", "grandmother,mom,dad,aunt", "half-sister, self", "He is our son. My husband and I have not been diagnosed with ADHD but are fairly certain he inherited it from both of us.", "Father / self", "Father and mother and child", "father has dyslexia. mother never tested but possibly could be", "father-mothet", "Father, mother", "Her parent", "Mom - Depression, Dad - OCD, Narcissistic Personality Disorder", "Mom, clinical depression. OCD, dad", "Mom, Dad, Sister", " Sibling and parent", "All immediate members have it Mom, Dad , older sister and Younger Brother", "biological parents and self", "birth mom and dad were both meth addicts", "both parents", "Both parents have adhd", "Both parents have ADHD and older brother has ADHD", "Mother/ father depression", "Mother/Father-Child", "Myself,dad,yungest son ", "parent", "Parent ", "Parent - depression", "Parent and grandmother ", "Parent has Dyslexia", "sibling, mother", "Siblings and mother", "Siblings,aunts uncles,mother,grandmother", "Mother ", "Mother  son", "Mother - depression", "Mother - PTSD, multiple auto immune", "Mother & possibly sibling evaluation in progress", "mother and brother", "Mother and daughter", "Mother and older brother ", "Mother and son", "mother daughter", "Mother delression", "Mother depression.", "Mother grandmother siblings", "Mother has ADD", "Mother has been diagnosed", "Mother has generalized anxiety and periodic depression ", "Mother PTSD", "Mother son", "Mother was diagnosed", "MOTHER- CHILD", "Mother- Depression", "Mother--depression", "mother-daughter", "mother-dauther", "Mother-Depression", "Mother-OCD", "Mother-son", "Mother, Aunt, Grandmother", "mother, bipolar 2", "Mother, brother ", "Mother, Brother ADD", "mother, grandmother", "Mother, I am not sure if I have, but I was told", "mother, ptsd", "Mother, sister", "Mother, sister ", "Mother, sisters and brother", "Mother,,great aunt,", "Mother,great aunt", "Mother,siblings,grandma,mother", "brother,sister,mom,grandma,aunt", "Child mom has problems cause brain tumor", "CHILD-MOTHER", "Child's mother", "child's mother mother also had learning disabilities in school", "Sister, likely mom", "Sister, mother", "Son-mother ", "Twin brother/sister/mother", "Grandfather mom's side, aunt mom's side. mom? ", "grandmother,mom", "half brother through mother", "Her mother me has it. ", "Her parent and her sibling", "his birth mom", "his birth mom and siblings", "His mother ", "His mother me", "Major Depressive Disorder, grandparent, mother", "mat. grandfather St. Vidas dance, mother depression", "Mim", "Mom", "Mom ", "mom and mom's family", "Mom has ocd", "Mom me anxiety", "Mom used to have psycho-motor absence epilepsy. Got cured.", "Mom, OCD dormant at this time", "Mom, possibly younger brother", "mom,grandma,brothers,aunt", "Mother", " mother", ",sister,aunt, mom,brothers,grandma", "Anxiety- mother", "Auntie mom sister ", "Bio mother", "Bio-mom closed adoption", "Biological mother", "biological mother and grandmother", "Birth mother", "Mother,uncle", "Mother. Sld", "Mother/  son", "Mother/ 2 brothers", "Mother/2brothere", "Mother/Daughter", "Mother/son", "motherself learning disabilty", "Mothmom,aunmother,great aunter,grandfather, grandmother,uncles and aunts ", "Mpm", "Parent mother"],
    
    'Parent: Father':["Parent w/ADHD", "Parent-child", "Parent, Uncle", "Parent/child", "Parents", "parents and siblings", "parents anxiety and depression", "Self, siblings, child/parent", "Self, sister and dad", "sibling and parent", "sibling, parent", "sibling, self", "siblings, parents", "mother - past mild depression & father - mild anxiety", "Mother & father", "mother & father major depression major anxitey", "mother and brother, anxiety.  father and mother, depression", "Mother and father", "Mother and father ", "mother and father Depression/ Anxiety", "Mother, father, sister", "Brother, Father, Mother", "Child both parents have adhd and older brother", "Child both parents have it", "Child/parent", "grandmother,mom,dad,aunt", "half-sister, self", "He is our son. My husband and I have not been diagnosed with ADHD but are fairly certain he inherited it from both of us.", "Father / self", "Father and mother and child", "father has dyslexia. mother never tested but possibly could be", "father-mothet", "Father, mother", "Her parent", "Mom - Depression, Dad - OCD, Narcissistic Personality Disorder", "Mom, clinical depression. OCD, dad", "Mom, Dad, Sister", " Sibling and parent", "All immediate members have it Mom, Dad , older sister and Younger Brother", "biological parents and self", "birth mom and dad were both meth addicts", "both parents", "Both parents have adhd", "Both parents have ADHD and older brother has ADHD", "Mother/ father depression", "Mother/Father-Child", "Myself,dad,yungest son ", "parent", "Parent ", "Parent - depression", "Parent and grandmother ", "Parent has Dyslexia", "Self and father", "Sibling and father", "Sibling, father", "sibling; father", "Brother and father", "Brother, Father, Grandmother, Aunt, Great Aunt", "Brother, father, uncle, grandfather ", "Brother, maybe father", "Brother, sister and father", "Child-father", "Child, Father and Siseter with Dyslexia", "Child's father", "Dad", "Dad and older brother have it", "Dad and uncle", "dad has ADD", "Sister and Father", "Sister, father", "sister, likely father undiagnosed", "Son - father", "Son and Father", "Son/ father  nephew/uncle ", "Son/Father", "we think father & grandmother but they were never tested", "Father/child", "Father/daughter", "Father/son", "father/son adhd", "Father\Son", "Fathers", "Fathet", "Dad has dyslexia very minor", "Dad is an early onset vascular dementia patient ", "Dad. But Mom is pretty good.", "Daf", "Depression father", "father", "Father ", "Father - BiPolar", "Father - Daughter", "Father ADD", "father and brother", "Father and likely brother", "Father and Siblings", "Father and sisters anxiety and depression ", "Father and Son", "Father dyslexia", "Father has ADD", "Father has adhd", "father has diagnosed ADD and slight dyslexia", "Father has dyslexia", "Father has Major Depressive Disorder", "father might have some learning disability", "Father son", "Father suspects dyslexia. Paterernal aunt has ALS, father's paternal uncle had also had ALS, grandfather had Parkinson's. A cousin might also have neurological dusorder", "Father undiagnosed ", "Father- Son", "Father-child", "F+A35ather-son", "father,  uncle", "Father, Aunt, Grandmother", "father, brothers", "Father, grandfather and uncle", "Father, Grandfather, Aunt, Cousin. ", "father, paternal grandfather, maternal uncle", "FATHER, UNCLE", "her dad suspected ADHD, not diagnosed; brother diagnosed with ADHD", "Her father and paternal grandmother have it. ", "His father", "His father believes that he had ADHD as a child still has it but was never officially diagnosed by a physician", "his father Darrell cost", "I am the father", "mat. grandmother, father, paternal half uncle", "Maybe his dad", " father", "Bio father", "Bio father ADHD/Impulsivity,  maternal grandfather ADD, Maternal cousin ADHD ", "Bio Father, perceptual learning challenges, possibly dyslexia, slow processing speed, ", "Bio-dad closed adoption", "Biological father", "Biological father ", "biological father, sister", "Birth father", "Birth father ", "Birth fathet", "My child's father has ADHD diag and possible autism", "My husband, Andrew's dad, is dyslexic, though he has never received a medical diagnosis.", "my husband/his dad has ADD", "My three daughthers and their dad has dyslexia.", "parent father"],
    
    'Sibling: Brother': ["parents and siblings", "Self, siblings, child/parent", "sibling and parent", "sibling, parent", "sibling, self", "siblings, parents", "mother and brother, anxiety.  father and mother, depression", "Brother, Father, Mother", "Child both parents have adhd and older brother", "Her parent", "Mom, Dad, Sister", " Sibling and parent", "All immediate members have it Mom, Dad , older sister and Younger Brother", "Both parents have ADHD and older brother has ADHD", "Mother/Father-Child", "Myself,dad,yungest son ", "Sibling and father", "Sibling, father", "sibling; father", "Brother and father", "Brother, Father, Grandmother, Aunt, Great Aunt", "Brother, father, uncle, grandfather ", "Brother, maybe father", "Brother, sister and father", "Dad and older brother have it", "Father/son", "father/son adhd", "Father\Son", "father and brother", "Father and likely brother", "Father and Siblings", "father, brothers", "her dad suspected ADHD, not diagnosed; brother diagnosed with ADHD", "sibling, mother", "Siblings and mother", "Siblings,aunts uncles,mother,grandmother", "Mother  son", "mother and brother", "Mother and older brother ", "Mother grandmother siblings", "Mother son", "MOTHER- CHILD", "Mother, brother ", "Mother, Brother ADD", "Mother, sisters and brother", "Mother,siblings,grandma,mother", "brother,sister,mom,grandma,aunt", "Twin brother/sister/mother", "half brother through mother", "Her parent and her sibling", "his birth mom and siblings", "Mom, possibly younger brother", "mom,grandma,brothers,aunt", ",sister,aunt, mom,brothers,grandma", "Mother/  son", "Mother/ 2 brothers", "Mother/2brothere", "Mother/son", "Sam has dyslexia; he is her 7 year old brother ", "Same brother", "Self and brother", "Self and sibling", "Self, brother", "Sibling", "sibling ", "Sibling - anxiety", "Sibling - brother", "Sibling and child", "Sibling with Down Syndrome ", "Sibling with temper dysregulation disorder and anxuety", "Sibling- Younger Brother", "Sibling-brother", "Sibling, child", "Sibling, son", "siblings", "Siblings ", "Brother ", "brother ADD", "brother and sister", "Brother diagnosed with stealth dyslexia", "Brother has mild, unmedicated ADHD", "Brother Trisomomy 21", "brother uncles cousin", "brother-- auditory processing ", "Brother, dysgraphia", "Brother, Sister", "Brother/sister", "brothers", "brothers ", "Calista's twin brother has Autism/ADHD", "Callie is the middle child so her older sister and younger brother have dyslexia. Father also believes he has it but was never diagnosed", "sister and brother", "Sister-Brother", "Sister, Brother", "Son and brother", "son, sibling", "Twin", "Twin ", "Twin brother", "twin brother ", "Twin brother, uncle", "younger Brother", "Younger brother has ADHD", "Younger brother Henry is 3", "fraternal twin brother", "Grandma,siblings", "Half brother", "half brother ", "half brother/self", "half sibling", "half-brother", "Fatehr, grandfather and sibling", "Her brother may had been dyslexic", "His brother ", "his siblings also FAS", "Liam is the brother to his little sister 5 years old", "Medio hermano", " Siblings ", "2 sibilings - 1 sister, 1 brother", "2brothers", "Biological brother", "both brothers", "brother", "My child", "My Child has ADHD.", "nephew and son", "Not sure if it counts but older brother was diagnose with ADD.", "older brother", "Older brother Jack is 11", "Older brother with depression", "Older siblings", "Oldest brother has mild ADHD", "Oldest sibling "],
    
    'Other Family Member':["Siblings,aunts uncles,mother,grandmother", "Mother grandmother siblings", "Mother,siblings,grandma,mother", "brother,sister,mom,grandma,aunt", ",sister,aunt, mom,brothers,grandma", "son, sibling", "Grandma,siblings", "Fatehr, grandfather and sibling", "Auntie mom sister ", "sister and daughter", "Sister daughter ", "sister/uncle", "Twin sister and first cousin", "Grandmother, Sister both have Cognitive Memory Disorder", "Maternal cousin, sister", "Nephew and sister ", "Brother, Father, Grandmother, Aunt, Great Aunt", "Brother, father, uncle, grandfather ", "Mother  son", "mom,grandma,brothers,aunt", "brother uncles cousin", "Son and brother", "Twin brother, uncle", "nephew and son", "Parent, Uncle", "grandmother,mom,dad,aunt", "Parent and grandmother ", "Dad and uncle", "Son - father", "Son and Father", "Son/ father  nephew/uncle ", "Son/Father", "we think father & grandmother but they were never tested", "Father suspects dyslexia. Paterernal aunt has ALS, father's paternal uncle had also had ALS, grandfather had Parkinson's. A cousin might also have neurological dusorder", "father,  uncle", "Father, Aunt, Grandmother", "Father, grandfather and uncle", "Father, Grandfather, Aunt, Cousin. ", "father, paternal grandfather, maternal uncle", "FATHER, UNCLE", "Her father and paternal grandmother have it. ", "mat. grandmother, father, paternal half uncle", "Bio father ADHD/Impulsivity,  maternal grandfather ADD, Maternal cousin ADHD ", "Mom,uncle", "Mother & possibly sibling evaluation in progress", "Mother and daughter", "Mother and son", "mother daughter", "mother-daughter", "mother-dauther", "Mother-son", "mother, grandmother", "Mother,,great aunt,", "Mother,great aunt", "Son-mother ", "Grandfather mom's side, aunt mom's side. mom? ", "grandmother,mom", "Major Depressive Disorder, grandparent, mother", "mat. grandfather St. Vidas dance, mother depression", "mom and mom's family", "biological mother and grandmother", "Mother,uncle", "Mothmom,aunmother,great aunter,grandfather, grandmother,uncles and aunts ", "Paternal grandmother", "Same uncle as above. ", "Second cousins", "cousin", "Cousin ", "Cousin and child herself", "Cousin and uncle", "Cousin on his mothers side had problems talking.", "Cousins", "Cousins ", "Sobrino", "son, nephew", "Stepson", "Uncle", "Uncle ", "Uncle twin and himself", "uncles", "Fathers uncle", "First cousin", "First Cousin ", "First cousin on maternal side", "First cousins, Uncle", "Grand son", "Grand uncle and aunt", "Grandchild and child", "Grandfather", "Grandfather ", "Grandfather - paternal", "Grandma and uncle", "Grandma,grandpa,uncle siblingings", "Grandmother", "Grandmother ", "Grandmother depression ", "grandmother on fathers side", "Grandpa", "grandson", "Grandson, nephew ", "granmother", "Great grandmother", "Father - paternal grandparents", "M Grandmother", "M Grandmother ", "M uncle", "Maternal uncle, great aunt", "Maternal uncle, paternal uncle", "Maternal Uncle, speech delay", "Meternal grandma", "Mom's Sister", "Moms sister", "Son", "Son ", "Son Late", " First cousin", "1st cousin", "2nd Cousin", "adopted siblings", "Aprexia ", "Aunt", "Aunt ", "aunt half-blood", "Aunt on fathers side", "aunt,grandmother", "aunt/nephew", "Aunt/niece", "Beatrix", "Biological egg donor and cousin.  ", "Mother's aunt", "My sons birth family is speckled with all of those issues", "My uncle her maternal great uncle. ", "Nephew", "Nephew ", "Nephew/uncle", "Other", "P Grandmother ", "Mother, Aunt, Grandmother"]}

1. loop through a dictionary called firstorder_relatives_dict
2. for each key (which represents a type of psychiatric disorder), check if the corresponding value (which is a list of responses that indicate the presence of that disorder in a first-order relative) is present in the dataframe first_order_relatives (by looping through a list of columns in first_order_relatives that correspond to different psychiatric disorders)
3. if a given response is present in a given column for a given first-order relative, the code sets a new column in first_order_relatives with a value of '1' indicating that the corresponding first-order relative has the given disorder. 

This is done for all the disorders and all the responses in the firstorder_relatives_dict.

In [87]:
for relative in firstorder_relatives_dict.keys():
    for col in ['psyc_famrel10_Other psychiatric and neurological disorders', 'psyc_famrel11_Major Depression', 'psyc_famrel5_Schizophrenia', 'psyc_famrel6_Bipolar Disorder', 'psyc_famrel9_Other developmental delay or intellectual disability', 'psyc_famrel4_Dyscalculia/math disability', 'psyc_famrel8_Epilepsy', 'psyc_famrel7_Fragile X syndrome', 'psyc_famrel1_Autism', 'psyc_famrel2_ADHD', 'psyc_famrel3_Dyslexia/Language disorders']:
        for rel_response in firstorder_relatives_dict[relative]:
            is_present = first_order_relatives[(first_order_relatives[f'participant_psycfam___{col.split("rel")[1].split("_")[0]}_{col.split("_")[-1]}'] != "0")][col].str.contains(rel_response, case=False, regex=True).any()
            if is_present:
                first_order_relatives.loc[first_order_relatives[col].str.contains(rel_response).fillna(False), f'{relative}_{col.split("_")[1]}_{col.split("_")[-1]}'] = '1'

/var/folders/rl/bc5_nh8n3910m9bytgmyn6l80000gn/T/ipykernel_62692/2184250883.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  first_order_relatives.loc[first_order_relatives[col].str.contains(rel_response).fillna(False), f'{relative}_{col.split("_")[1]}_{col.split("_")[-1]}'] = '1'


In [102]:
# display(first_order_relatives)
print(first_order_relatives.columns[-40:])

Index(['Parent: mother_famrel10_Other psychiatric and neurological disorders',
       'Parent: mother_famrel11_Major Depression',
       'Parent: mother_famrel5_Schizophrenia',
       'Parent: mother_famrel6_Bipolar Disorder',
       'Parent: mother_famrel9_Other developmental delay or intellectual disability',
       'Parent: mother_famrel4_Dyscalculia/math disability',
       'Parent: mother_famrel8_Epilepsy', 'Parent: mother_famrel1_Autism',
       'Parent: mother_famrel2_ADHD',
       'Parent: mother_famrel3_Dyslexia/Language disorders',
       'Parent: Father_famrel10_Other psychiatric and neurological disorders',
       'Parent: Father_famrel5_Schizophrenia',
       'Parent: Father_famrel6_Bipolar Disorder',
       'Parent: Father_famrel9_Other developmental delay or intellectual disability',
       'Parent: Father_famrel4_Dyscalculia/math disability',
       'Parent: Father_famrel8_Epilepsy', 'Parent: Father_famrel1_Autism',
       'Parent: Father_famrel2_ADHD',
       'Parent: 

In [89]:
first_order_relatives_cols_opposite = {'participant_psycfam___1':'participant_psycfam___1_Autism',
                                       'participant_psycfam___2':'participant_psycfam___2_ADHD',
                                       'participant_psycfam___3':'participant_psycfam___3_Dyslexia/Language disorders',
                                       'participant_psycfam___4':'participant_psycfam___4_Dyscalculia/math disability',
                                       'participant_psycfam___5':'participant_psycfam___5_Schizophrenia',
                                       'participant_psycfam___6':'participant_psycfam___6_Bipolar Disorder',
                                       'participant_psycfam___7':'participant_psycfam___7_Fragile X syndrome',
                                       'participant_psycfam___8':'participant_psycfam___8_Epilepsy',
                                       'participant_psycfam___11':'participant_psycfam___11_Major Depression',
                                       'participant_psycfam___9':'participant_psycfam___9_Other developmental delay or intellectual disability',
                                       'participant_psycfam___10':'participant_psycfam___10_Other psychiatric and neurologic disorders',
                                       'psyc_famrel1':'psyc_famrel1_Autism',
                                       'psyc_famrel2':'psyc_famrel2_ADHD',
                                       'psyc_famrel3':'psyc_famrel3_Dyslexia/Language disorders',
                                       'psyc_famrel4':'psyc_famrel4_Dyscalculia/math disability',
                                       'psyc_famrel5':'psyc_famrel5_Schizophrenia',
                                       'psyc_famrel6':'psyc_famrel6_Bipolar Disorder',
                                       'psyc_famrel7':'psyc_famrel7_Fragile X syndrome',
                                       'psyc_famrel8':'psyc_famrel8_Epilepsy',
                                       'psyc_famrel9':'psyc_famrel9_Other developmental delay or intellectual disability',
                                       'psyc_famrel10':'psyc_famrel10_Other psychiatric and neurological disorders',
                                       'psyc_famrel12':'psyc_famrel12_Major Depression',
                                       'psyc_extra':'psyc_extra_general'}


first_order_relatives_cols_opposite_switched = {v: k for k, v in first_order_relatives_cols_opposite.items()}

print(first_order_relatives_cols_opposite_switched)
first_order_relatives.rename(columns={'participant_psycfam___1_Autism': 'participant_psycfam___1', 'participant_psycfam___2_ADHD': 'participant_psycfam___2', 'participant_psycfam___3_Dyslexia/Language disorders': 'participant_psycfam___3', 'participant_psycfam___4_Dyscalculia/math disability': 'participant_psycfam___4', 'participant_psycfam___5_Schizophrenia': 'participant_psycfam___5', 'participant_psycfam___6_Bipolar Disorder': 'participant_psycfam___6', 'participant_psycfam___7_Fragile X syndrome': 'participant_psycfam___7', 'participant_psycfam___8_Epilepsy': 'participant_psycfam___8', 'participant_psycfam___11_Major Depression': 'participant_psycfam___11', 'participant_psycfam___9_Other developmental delay or intellectual disability': 'participant_psycfam___9', 'participant_psycfam___10_Other psychiatric and neurologic disorders': 'participant_psycfam___10', 'psyc_famrel1_Autism': 'psyc_famrel1', 'psyc_famrel2_ADHD': 'psyc_famrel2', 'psyc_famrel3_Dyslexia/Language disorders': 'psyc_famrel3', 'psyc_famrel4_Dyscalculia/math disability': 'psyc_famrel4', 'psyc_famrel5_Schizophrenia': 'psyc_famrel5', 'psyc_famrel6_Bipolar Disorder': 'psyc_famrel6', 'psyc_famrel7_Fragile X syndrome': 'psyc_famrel7', 'psyc_famrel8_Epilepsy': 'psyc_famrel8', 'psyc_famrel9_Other developmental delay or intellectual disability': 'psyc_famrel9', 'psyc_famrel10_Other psychiatric and neurological disorders': 'psyc_famrel10', 'psyc_famrel12_Major Depression': 'psyc_famrel12', 'psyc_extra_general': 'psyc_extra'},inplace=True)


{'participant_psycfam___1_Autism': 'participant_psycfam___1', 'participant_psycfam___2_ADHD': 'participant_psycfam___2', 'participant_psycfam___3_Dyslexia/Language disorders': 'participant_psycfam___3', 'participant_psycfam___4_Dyscalculia/math disability': 'participant_psycfam___4', 'participant_psycfam___5_Schizophrenia': 'participant_psycfam___5', 'participant_psycfam___6_Bipolar Disorder': 'participant_psycfam___6', 'participant_psycfam___7_Fragile X syndrome': 'participant_psycfam___7', 'participant_psycfam___8_Epilepsy': 'participant_psycfam___8', 'participant_psycfam___11_Major Depression': 'participant_psycfam___11', 'participant_psycfam___9_Other developmental delay or intellectual disability': 'participant_psycfam___9', 'participant_psycfam___10_Other psychiatric and neurologic disorders': 'participant_psycfam___10', 'psyc_famrel1_Autism': 'psyc_famrel1', 'psyc_famrel2_ADHD': 'psyc_famrel2', 'psyc_famrel3_Dyslexia/Language disorders': 'psyc_famrel3', 'psyc_famrel4_Dyscalculia

/var/folders/rl/bc5_nh8n3910m9bytgmyn6l80000gn/T/ipykernel_62692/875042320.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  first_order_relatives.rename(columns={'participant_psycfam___1_Autism': 'participant_psycfam___1', 'participant_psycfam___2_ADHD': 'participant_psycfam___2', 'participant_psycfam___3_Dyslexia/Language disorders': 'participant_psycfam___3', 'participant_psycfam___4_Dyscalculia/math disability': 'participant_psycfam___4', 'participant_psycfam___5_Schizophrenia': 'participant_psycfam___5', 'participant_psycfam___6_Bipolar Disorder': 'participant_psycfam___6', 'participant_psycfam___7_Fragile X syndrome': 'participant_psycfam___7', 'participant_psycfam___8_Epilepsy': 'participant_psycfam___8', 'participant_psycfam___11_Major Depression': 'participant_psycfam___11', 'participant_ps

In [90]:
first_order_relatives.to_csv(output_filename_prefix+'_updated_firstorder_relatives_w_dict.csv')

In [91]:
diagnosis1={
      '1':{'ADHD/ADD':{'add','add ','adhd ', 'adhd', 'attention deficit','add/adhd','ADHD','ADHD/ADD','adhd/add','behavioral/ add'}
     },
    '2': {'Aggression': {'aggression', 'aggressive', 'explosive', 'anger'}
          },
    '3': {'Anxiety (Generalized, Separation, Social, Panic, Attachment)': {'anx', 'anxious', 'anxiety', 'anxiety nos', 'nos anxiety', 'transition anxiety', 'reactive attachment disorder', 'panic disorder', 'social anxiety', 'social anxiety', 'separation anxiety', 'anxiety (gad-nos)'}
          },
    '4': {'ASD/Autism': {'asd/autism', 'asd', 'hf asd', 'high functioning autism', 'asd', 'autism', 'spectrum'}},
    '5': {'Aspergers': {'asperger\'s', 'aspergers', 'asberger', 'aspie'}
          },
    '6': {'Bipolar disorder': {'bipolar disorder', 'bipolar', 'manic', 'bi-polar', 'bpd', 'Bipolar'}
          },
    '7': {'Depression/Major Depression Disorder (MDD)': {'depressive', 'dep', 'depression', 'MDD', 'depress', 'major depression', 'depressive'}
          },
    '8': {'Emotional Disturbance Disorder/Other Mood Disorder': {'mood disorder', 'mood disorder - nos', 'emotional disturbance disorder'}
          },
    '9': {'Obssessive Compulsive Disorder (OCD)': {'ocd', 'obsessive', 'compulsive', 'OCD'}
          },
    '10': {'Oppositional Defiant Disorder (ODD)': {'oppositional defiant disorder (odd)', 'odd', 'oppositional', 'defiant'}
           },
    '11': {'Pervasive Developmental Disorder Not Otherwise Specified (PDD-NOS)': {'pdd-nos', 'PDD'}
           },
    '12': {'Phobia/specific fear (e.g., claustrophobia)': {'claustrophobic', 'phobia/specific fear', 'phobia'}
           },
    '13': {'PTSD/Trauma/Adjustment Disorder': {'trauma/ptsd', 'adjustment disorder', 'ptsd', 'trauma', 'ptsd', 'adjustment'}
           },
    '14': {'Selective Mutism': {'selective mutism', 'mutism'}
           },
    '15': {'Sensory Integration/Processing Disorder': {'sensory processing', 'sensory', 'sensory processing disorder', 'sensory integration', 'sensory disorder'}
           },
    '16': {'Tic/Tourettes': {'tourette', 'tic'}
           },
    '17': {'Trichiyillomania': {'trichiyillomania'}
           },
    '98': {'NA (not psychological; not listed above)': {'other', 'ed', 'dysgraphia', 'developmental', 'communication disorder'}
           }
}


In [ ]:
d = {'1': {'participant_meds_thyroid': {'cytomel', 'synthroid', 'levothyroxine', 'levoxl', 'levothroidwesthroid', 'liothyronine', 'armourthyroid', 'levothroid', 'propanolol', 'inderalla', 'innopranxl', 'hemangeol', 'inderal', 'yto', 'ynth', 'vothy', 'evox', 'roidwe', 'iothy', 'mourt', 'evothr', 'opan', 'ralla', 'nopra', 'geol', 'inde', 'thyroid'}
           },
     '2': {'participant_meds_blood_thinners': {'atripla', 'stribild', 'complera', 'triumeq', 'heparin', 'lovenox', 'fragmin', 'eliquis', 'pradaxa', 'xarelto', 'plavix', 'warfarin', 'coumadin', 'ipla', 'bild', 'compl', 'umeq', 'hepa', 'love', 'agmi', 'iqui', 'adax', 'xare', 'plav', 'warf', 'umad'}
           },
     '3': {'participant_meds_ssri_MDD': {'pexeva', 'prozac', 'celexa', 'lexapro', 'luvox', 'paxil', 'zoloft', 'cymbalta', 'effexor', 'remeron', 'wellbutrin', 'anafranil', 'norpramin', 'pamelor', 'sinequan', 'surmontil', 'minipress', 'citalopram', 'escitalopram', 'fluoxetine', 'paroxetine', 'sertraline', 'sert', 'duloxetine', 'venlafaxine', 'mirtazapine', 'bupropion', 'clomipramine', 'amitriptyline', 'desipramine', 'nortriptyline', 'doxepin', 'trimipramine', 'prazosin', 'pine', 'pam', 'mine', 'pram'}
           },
     '4': {'participant_meds_psychotic': {'risperdal', 'risperidone', 'risp', 'abilify', 'abil', 'fy' 'zyprexa', 'seroquel', 'geodon', 'therazine', 'prolixin', 'haldol', 'moban', 'navane', 'stelazine', 'loxitane', 'riperidone', 'aripirazole', 'olanzapine', 'quetiapine', 'ziprasidone', 'chlorpromazine', 'fluphenazine', 'haloperidol', 'molindone', 'thiothixene', 'trifluoperazine', 'loxapine', 'adasuve', 'ormazine', 'permitil', 'abil', 'ypre', 'roqu', 'eod', 'heraz', 'olix', 'aldo', 'oba', 'ava', 'elaz', 'xit', 'ripe', 'pira', 'lanz', 'etia', 'ipras', 'oma', 'luph', 'operid', 'olin', 'hix', 'iflu', 'loxa', 'dasu', 'rmaz', 'ermi'}
           },
     '5': {'participant_meds_anti_convulsant': {'depakene', 'depakote', 'tegretol', 'lithobid', 'eskalith', 'oxcarbazepine', 'lamotrigine', 'lamot', 'lithium', 'valproicacid', 'divalproex', 'trileptal', 'lamictal', 'carbamazepine', 'pake', 'pako', 'gret', 'thob', 'skal', 'xcarb', 'motri', 'thium', 'oicac/lproe', 'ilep', 'amic'}
           },
     '6': {'participant_meds_attention_add': {'amphetamine', 'methylphenidate', 'nonstimulants', 'adderall', 'focaline', 'strattera', 'intuniv', 'dexedrine', 'dextrostat', 'vyvanse', 'methylin', 'ritalin', 'metadate', 'concerta', 'quillivant', 'daytrana', 'ritalin', 'hetam', 'ylphen', 'nonsti', 'adde', 'foca', 'strat', 'univ', 'dexed', 'dextros', 'yvans', 'hylin', 'rita', 'tadat', 'conc', 'illiv', 'ytran', 'rit'}
           },
     '7': {'participant_meds_anti_anx': {'alprazolam', 'ativan', 'lorazepam', 'xanax', 'clonazepam', 'valium', 'prazol', 'tiva', 'oraze', 'anax', 'onaze', 'alium'}
           },
     '8': {'participant_meds_sleep': {'melatonin', 'ambien', 'eszopiclone', 'ramelteon', 'triazolam', 'zaleplon', 'aolpidem', 'trazadone', 'lunesta', 'halcion', 'ambien', 'rozerem', 'sonata', 'zopic', 'elteo', 'iazo', 'alepl', 'olpi', 'azad', 'unest', 'alcio', 'mbien', 'ozere', 'onata'}
           },
     '9': {'chronic_pain_meds': {'buprenorphine', 'renorp', 'butorphanol', 'torph', 'codeine', 'ein', 'hydrocodone', 'rocod', 'hydromorphone', 'romorp', 'oxymorphone', 'ymorp', 'propxyphene', 'pyxp', 'tapentadol', 'ntado', 'tramadol', 'ramad', 'dilaudid', 'laudi', 'acetaminophen', 'cetami', 'hydrostat', 'ydros', 'exalgo', 'xalg', 'numorphan', 'umorp', 'contanal', 'ontan', 'darvon', 'arvon', 'ultracet', 'ltrac', 'nucynta', 'ucynt', 'buprenex', 'upren', 'butranspatch', 'ransp', 'stadol', 'tadol', 'morphine', 'rphin', 'nalbuphine', 'lbuph', 'oxycodone', 'ycodo', 'pentazocine', 'ntazo', 'diclofenac', 'clofe', 'meloxicam', 'loxic', 'naproxen', 'eproxe', 'gabapentin', 'bapent', 'pregabalin', 'egaba', 'fentanyl', 'ntany', 'meperidine', 'eridi', 'methadone', 'thado', 'levorphanol', 'vorphan', 'levo-dromoran', 'romora', 'demerol', 'emero', 'dolophine', 'olophi', 'astramorph', 'stramo', 'nubain', 'ubai', 'oxycontin', 'xycont', 'talwin', 'alwi', 'cataflam', 'atafla', 'mobic', 'obic', 'aleve', 'leve', 'neurontin', 'uronti', 'lyrica', 'yric', 'duragesic', 'urage', 'methadose', 'ethad', 'voltaren', 'oltar', 'msir', 'cont', 'kadian', 'adian', 'duramorph', 'uramor', 'avinza', 'vinza', 'roxicodone', 'oxicodo', 'oxecta', 'xecta', 'zipsor', 'ipsor', 'naprelan', 'aprela', 'anaproct', 'anaproc', 'naproc', 'gralise', 'alise', 'naprosyn', 'aprosy', 'butranspatch', 'butran', 'ibuprofen', 'advil', 'tylenol', 'vil', 'ylen', 'uprof'}
           },
     '10': {'participant_adhd_stimulants_need_washout': {'aptensio', 'ptens', 'aptensio xr', 'concerta', 'erta', 'certa', 'daytrana', 'ytran', 'focalin', 'focalin xr', 'ocal', 'metadate', 'metadate er', 'tadat', 'methylin', 'hylin', 'quillivant', 'quillichew', 'illiv', 'ritalin', 'ritalin la', 'rita', 'adderall', 'adderall xr', 'add', 'ral', 'adde', 'dexedrine', 'edrine', 'dyanavel', 'dyanevel xr', 'ynave', 'evekeo', 'veke', 'procentra', 'rocen', 'vyvanse', 'vyv', 'amphetamine', 'dextroamphetamine', 'hetam', 'methylphenidate', 'ylphen', 'dexmethylphenidate', 'xmethy', 'lisdexamfetamine', 'lisd'}
            },
     '11': {'participant_adhd_non_stim': {'intuniv', 'univ', 'kapvay', 'apva', 'wellbutrin', 'lbutri', 'viloxazine', 'vilo', 'guanfacine', 'gua', 'fancine', 'uanf', 'clonidine', 'clon', 'buproprion', 'prion'}
            },
     '12': {'participant_adhd_non_stim_nowashout': {'strattera', 'trat', 'atomoxetine', 'atomoxetine hcl', 'mox'}
            },
     '13': {'participant_meds_allergy_asthma': {'asthma', 'albuterol', 'alb', 'nasonex', 'cetrizine', 'cetri', 'zine', 'nasal', 'spray', 'inhaler', 'flonase', 'montelukast', 'singulair', 'buter', 'inh', 'flo', 'mont', 'ulair', 'zyrtec', 'zyr', 'claritin', 'clar', 'zyzal', 'sinus', 'allergies', 'benadryl', 'dryl', 'loratidine', 'qvar'}
            },
     '14': {'antibiotics': {'antibiotics', 'antib', 'biotic', 'azithro', 'mycine', 'vir', 'thromyc'}
            },
     '15': {'participant_meds_skin': {'ecz', 'ecema', 'retinoid', 'retin-a', 'tacrolimus', 'rolim', 'differin', 'diff', 'accutane', 'accu', 'steroid'}
            }
     }


In [ ]:
registration_form_data_export=scsnl_db

In [ ]:
for key in d:
    for med_group in d[key]:
        for i, med in enumerate(d[key][med_group]):
            # Check if the med value is present in the psycmeds_desc column
            is_present = registration_form_data_export[(registration_form_data_export['participant_psycmeds'] != "no")
                                                       & (registration_form_data_export['psycmeds_desc'] != "")]['psycmeds_desc'].str.contains(med, case=False, regex=True).any()
            is_present_meds = registration_form_data_export[(registration_form_data_export['participant_meds'] != "no")
                                                       & (registration_form_data_export['recentmeds_desc'] != "")]['recentmeds_desc'].str.contains(med, case=False, regex=True).any()
            other_ispresent_meds=registration_form_data_export[(registration_form_data_export['participant_psyccond'] != "no")
                                                                & (registration_form_data_export['psyccond_desc'] != "")]['psyccond_desc'].str.contains(med, case=False, regex=True).any()      
            specific_psych=registration_form_data_export[(registration_form_data_export['participant_psycspec'] != "no")
                                                                & (registration_form_data_export['psycspec_desc'] != "")]['psycspec_desc'].str.contains(med, case=False, regex=True).any()      
                              
            if is_present:
                # Set the new column to the value of key where med is present
                registration_form_data_export.loc[registration_form_data_export['psycmeds_desc'].str.contains(med).fillna(False), med_group]=key
            if is_present_meds:
                registration_form_data_export.loc[registration_form_data_export['recentmeds_desc'].str.contains(med).fillna(False), med_group]=key
            if other_ispresent_meds:
                registration_form_data_export.loc[registration_form_data_export['psyccond_desc'].str.contains(med).fillna(False), med_group]=key
            if specific_psych:
                registration_form_data_export.loc[registration_form_data_export['psycspec_desc'].str.contains(med).fillna(False), med_group]=key

In [ ]:
for key in diagnosis1.keys():
    for dx_group in diagnosis1[key]:
        for i, dx in enumerate(diagnosis1[key][dx_group]):
            # Check if the med value is present in the psycmeds_desc column
           
            is_present=registration_form_data_export[(registration_form_data_export['participant_psyccond'] != "no")
                                                                & (registration_form_data_export['psyccond_desc'] != "")]['psyccond_desc'].str.contains(dx, case=False, regex=True).any()
            anxiety_dep_sz_specific=registration_form_data_export[(registration_form_data_export['participant_psycspec'] != "no")
                                                     & (registration_form_data_export['psycspec_desc'] != "")]['psycspec_desc'].str.contains(dx, case=False, regex=True).any()                                         
            if is_present:
                # Set the new column to the value of key where med is present
                registration_form_data_export.loc[registration_form_data_export['psyccond_desc'].str.contains(dx).fillna(False), dx_group]=key
            if anxiety_dep_sz_specific:
                registration_form_data_export.loc[registration_form_data_export['psycspec_desc'].str.contains(dx).fillna(False), dx_group]=key
            else:
                registration_form_data_export.loc[registration_form_data_export['psycspec_desc'].str.contains(dx).fillna(False), 'other']='1'
                
# display(registration_form_data_export)

In [ ]:

registration_form_data_export.rename(columns={'ADHD/ADD':'psyc_cond_checkbox___1',
'Aggression':'psyc_cond_checkbox___2',
'Anxiety (Generalized, Separation, Social, Panic, Attachment)':'psyc_cond_checkbox___3',
'ASD/Autism':'psyc_cond_checkbox___4',
'Aspergers':'psyc_cond_checkbox___5',
'Bipolar disorder':'psyc_cond_checkbox___6',
'Depression/Major Depression Disorder (MDD)':'psyc_cond_checkbox___7',
'Emotional Disturbance Disorder/Other Mood Disorder':'psyc_cond_checkbox___8',
'Obssessive Compulsive Disorder (OCD)':'psyc_cond_checkbox___9',
'Oppositional Defiant Disorder (ODD)':'psyc_cond_checkbox___10',
'Pervasive Developmental Disorder Not Otherwise Specified (PDD-NOS)':'psyc_cond_checkbox___11',
'Phobia/specific fear (e.g., claustrophobia)':'psyc_cond_checkbox___12',
'PTSD/Trauma/Adjustment Disorder':'psyc_cond_checkbox___13',
'Selective Mutism':'psyc_cond_checkbox___14',
'Sensory Integration/Processing Disorder':'psyc_cond_checkbox___15',
'Tic/Tourettes':'psyc_cond_checkbox___16',
'Trichiyillomania':'psyc_cond_checkbox___17',
'other':'psyc_cond_checkbox___99',
'NA (not psychological; not listed above)':'psyc_cond_checkbox___98'}).to_csv('scsnl_db_psych_fixed_and_specific_psych.csv')

In [ ]:
registration_form_data_export.to_csv(output_filenamne_prefix+'psych_fixed_and_specific_psych.csv')

In [ ]:
# for relative in firstorder_relatives_dict.keys():
#     for rel_response in firstorder_relatives_dict[relative]:
#             is_present_other= first_order_relatives[(first_order_relatives['participant_psycfam___10_Other psychiatric and neurologic disorders'] != "0")]['psyc_famrel10_Other psychiatric and neurological disorders'].str.contains(rel_response,case=False,regex=True).any()
            
#             is_present_mdd= first_order_relatives[(first_order_relatives['participant_psycfam___11_Major Depression'] != "0")]['psyc_famrel12_Major Depression'].str.contains(rel_response,case=False,regex=True).any()
            
#             is_present_sz= first_order_relatives[(first_order_relatives['participant_psycfam___5_Schizophrenia'] != "0")]['psyc_famrel5_Schizophrenia'].str.contains(rel_response,case=False,regex=True).any()
            
#             is_present_bpdx=first_order_relatives[(first_order_relatives['participant_psycfam___6_Bipolar Disorder'] != "0")]['psyc_famrel6_Bipolar Disorder'].str.contains(rel_response,case=False,regex=True).any()
            
#             is_present_other_dev= first_order_relatives[(first_order_relatives['participant_psycfam___9_Other developmental delay or intellectual disability'] != "0")]['psyc_famrel9_Other developmental delay or intellectual disability'].str.contains(rel_response,case=False,regex=True).any()
            
#             is_present_dyscalcy= first_order_relatives[(first_order_relatives['participant_psycfam___4_Dyscalculia/math disability'] != "0")]['psyc_famrel4_Dyscalculia/math disability'].str.contains(rel_response,case=False,regex=True).any()
            
#             is_present_epilepsy= first_order_relatives[(first_order_relatives['participant_psycfam___8_Epilepsy'] != "0")]['psyc_famrel8_Epilepsy'].str.contains(rel_response,case=False,regex=True).any()
            
#             is_present_fx= first_order_relatives[(first_order_relatives['participant_psycfam___7_Fragile X syndrome'] != "0")]['psyc_famrel7_Fragile X Syndrome'].str.contains(rel_response,case=False,regex=True).any()
            
#             is_present_asd= first_order_relatives[(first_order_relatives['participant_psycfam___1_Autism'] != "0")]['psyc_famrel1_Autism'].str.contains(rel_response,case=False,regex=True).any()
            
#             is_present_adhd= first_order_relatives[(first_order_relatives['participant_psycfam___2_ADHD'] != "0")]['psyc_famrel2_ADHD'].str.contains(rel_response,case=False,regex=True).any()
            
#             is_present_dys= first_order_relatives[(first_order_relatives['participant_psycfam___3_Dyslexia/Language disorders'] != "0")]['psyc_famrel3_Dyslexia/Language disorders'].str.contains(rel_response,case=False,regex=True).any()
            
            
#             if is_present_bpdx:
#                 first_order_relatives.loc[first_order_relatives['psyc_famrel6_Bipolar Disorder'].str.contains(rel_response).fillna(False), relative+'_psyc_famrel6_bipolar']='1'

#             if is_present_mdd:
#                 first_order_relatives.loc[first_order_relatives['psyc_famrel12_Major Depression'].str.contains(rel_response).fillna(False), relative+'_psyc_famrel11_mdd']='1'
                
#             if is_present_other:
#                 first_order_relatives.loc[first_order_relatives['psyc_famrel10_Other psychiatric and neurological disorders'].str.contains(rel_response).fillna(False), relative+'_psyc_famrel10_other_psych_neuro_dx']='1'

#             if is_present_sz:
#                 first_order_relatives.loc[first_order_relatives['psyc_famrel5_Schizophrenia'].str.contains(rel_response).fillna(False), relative+'_psyc_famrel5_sz']='1'
                
                
#             if is_present_dys:            
#                 first_order_relatives.loc[first_order_relatives['psyc_famrel3_Dyslexia/Language disorders'].str.contains(rel_response).fillna(False), relative+'_psyc_famrel3_dyslexia_language_dx']='1'
                
#             if is_present_adhd:
#                 first_order_relatives.loc[first_order_relatives['psyc_famrel2_ADHD'].str.contains(rel_response).fillna(False), relative+'_psyc_famrel2_adhd']='1'
                
#             if is_present_asd:
#                 first_order_relatives.loc[first_order_relatives['psyc_famrel1_Autism'].str.contains(rel_response).fillna(False), relative+'_psyc_famrel1_asd']='1'
                
#             if is_present_fx:
#                 first_order_relatives.loc[first_order_relatives['psyc_famrel7_Fragile X Syndrome'].str.contains(rel_response).fillna(False), relative+'_psyc_famrel7_fragile_x_syndrome']='1'
                
#             if is_present_epilepsy:
#                 first_order_relatives.loc[first_order_relatives['psyc_famrel8_Epilepsy'].str.contains(rel_response).fillna(False), relative+'_psyc_famrel8_epilepsy']='1'
                
#             if is_present_dyscalcy:
#                 first_order_relatives.loc[first_order_relatives['psyc_famrel4_Dyscalculia/math disability'].str.contains(rel_response).fillna(False), relative+'_psyc_famrel4_math_dyscalculia_dx']='1'
                
#             if is_present_other_dev:
#                 first_order_relatives.loc[first_order_relatives['psyc_famrel9_Other developmental delay or intellectual disability'].str.contains(rel_response).fillna(False), relative+'_psyc_famrel9_other_intel_dis_dev_delay']='1'
                


# test=first_order_relatives.head(15)

/var/folders/rl/bc5_nh8n3910m9bytgmyn6l80000gn/T/ipykernel_62692/681293894.py:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  first_order_relatives.loc[first_order_relatives['psyc_famrel2_ADHD'].str.contains(rel_response).fillna(False), relative+'_psyc_famrel2_adhd']='1'
/var/folders/rl/bc5_nh8n3910m9bytgmyn6l80000gn/T/ipykernel_62692/681293894.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  first_order_relatives.loc[first_order_relatives['psyc_famrel6_Bipolar Disorder'].str.contains(rel_response).fillna(False), relative+'_psyc_famrel6_bipolar']='1'
/var/folders/rl/bc5_nh8n3910m9bytgmyn6l80000gn/T/ipykernel_6269